In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.decomposition import TruncatedSVD
import random
import seaborn as sns
import os.path as path
import os
import matplotlib
import matplotlib.font_manager
import matplotlib.pyplot as plt # graphs plotting
from Bio import SeqIO # some BioPython that will come in handy
#matplotlib inline
import numpy
import csv 

import timeit
from matplotlib import rc
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.preprocessing import StandardScaler

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn import metrics
from sklearn import svm

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn import metrics

import pandas as pd
import sklearn
from sklearn import preprocessing
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler  
from sklearn.neural_network import MLPClassifier 
from sklearn.metrics import classification_report, confusion_matrix 

from sklearn.neighbors import KNeighborsClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

from pandas import DataFrame

from sklearn.model_selection import KFold 
from sklearn.model_selection import RepeatedKFold

from sklearn.metrics import confusion_matrix

from numpy import mean


from itertools import cycle

from sklearn import svm, datasets
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from numpy import interp
from sklearn.metrics import roc_auc_score

import statistics

from sklearn.cluster import KMeans

from sklearn.datasets import load_digits
from sklearn.decomposition import KernelPCA

import math
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import confusion_matrix

# for Arial typefont
matplotlib.rcParams['font.family'] = 'Arial'


## for Palatino and other serif fonts use:
# rc('font',**{'family':'serif','serif':['Palatino']})
# matplotlib.rcParams['mathtext.fontset'] = 'cm'

## for LaTeX typefont
# matplotlib.rcParams['mathtext.fontset'] = 'stix'
# matplotlib.rcParams['font.family'] = 'STIXGeneral'

## for another LaTeX typefont
# rc('font',**{'family':'sans-serif','sans-serif':['Helvetica']})

# rc('text', usetex = True)

print("Packages imported")

Packages imported


# Loading Data and Boruta Feature Selection

In [5]:
df = pd.read_csv("cohort_all_features_capped.csv")
labels = df["delirium_label"]
X_df = df.drop(columns="delirium_label")

rng = np.random.default_rng(42)
X_shadow = X_df.apply(lambda col: rng.permutation(col.values))
X_shadow.columns = ["shadow_" + c for c in X_df.columns]
X_boruta = pd.concat([X_df, X_shadow], axis=1)

y = df["delirium_label"]
from sklearn.ensemble import RandomForestRegressor

### fit a random forest (suggested max_depth between 3 and 7)
forest = RandomForestRegressor(max_depth=5, random_state=42)
forest.fit(X_boruta, y)
### store feature importances
feat_imp_X = forest.feature_importances_[:len(X_df.columns)]
feat_imp_shadow = forest.feature_importances_[len(X_df.columns):]
### compute hits
hits = feat_imp_X > feat_imp_shadow.max()
selected_features = X_df.columns[hits].tolist()

# export to CSV
pd.DataFrame({"feature": selected_features}).to_csv("boruta_selected_features.csv", index=False)

print(f"{len(selected_features)} features selected")
print(selected_features)
final_boruta_data = X_df[selected_features]

KeyboardInterrupt: 

# Classification Functions

In [6]:
from sklearn.metrics import average_precision_score, precision_recall_curve

def roc_auc_score_multiclass(actual_class, pred_class, average="macro"):
    unique_class = set(actual_class)
    roc_auc_dict = {}
    for per_class in unique_class:
        other_class = [x for x in unique_class if x != per_class]
        new_actual_class = [0 if x in other_class else 1 for x in actual_class]
        new_pred_class = [0 if x in other_class else 1 for x in pred_class]
        roc_auc = roc_auc_score(new_actual_class, new_pred_class, average=average)
        roc_auc_dict[per_class] = roc_auc
    check = pd.DataFrame(roc_auc_dict.items())
    return mean(check)


##########################  SVM Classifier  ################################
def svm_fun(X_train, y_train, X_test, y_test):
    X_train = preprocessing.scale(X_train)
    X_test = preprocessing.scale(X_test)

    start = timeit.default_timer()
    clf = LinearSVC(class_weight='balanced', max_iter=5000)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    stop = timeit.default_timer()
    time_new = stop - start

    svm_acc = metrics.accuracy_score(y_test, y_pred)
    svm_prec = metrics.precision_score(y_test, y_pred, average='weighted')
    svm_recall = metrics.recall_score(y_test, y_pred, average='weighted')
    svm_f1_weighted = metrics.f1_score(y_test, y_pred, average='weighted')
    svm_f1_macro = metrics.f1_score(y_test, y_pred, average='macro')

    # was: y_prob = clf.predict_proba(X_test)[:, 1]
    y_prob = clf.decision_function(X_test)
    macro_roc_auc_ovo = roc_auc_score(y_test, y_prob)
    auprc = average_precision_score(y_test, y_prob)

    check = [svm_acc, svm_prec, svm_recall, svm_f1_weighted, svm_f1_macro,
             macro_roc_auc_ovo, auprc, time_new]
    return check


##########################  NB Classifier  ################################
def gaus_nb_fun(X_train, y_train, X_test, y_test):

    start = timeit.default_timer()
    gnb = GaussianNB()
    y_pred = gnb.fit(X_train, y_train).predict(X_test)
    stop = timeit.default_timer()
    time_new = stop - start

    NB_acc = metrics.accuracy_score(y_test, y_pred)
    NB_prec = metrics.precision_score(y_test, y_pred, average='weighted')
    NB_recall = metrics.recall_score(y_test, y_pred, average='weighted')
    NB_f1_weighted = metrics.f1_score(y_test, y_pred, average='weighted')
    NB_f1_macro = metrics.f1_score(y_test, y_pred, average='macro')

    y_prob = gnb.predict_proba(X_test)[:, 1]
    macro_roc_auc_ovo = roc_auc_score(y_test, y_prob)
    auprc = average_precision_score(y_test, y_prob)

    check = [NB_acc, NB_prec, NB_recall, NB_f1_weighted, NB_f1_macro,
             macro_roc_auc_ovo, auprc, time_new]
    return check


##########################  MLP Classifier  ################################
def mlp_fun(X_train, y_train, X_test, y_test):

    start = timeit.default_timer()
    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train = scaler.transform(X_train)
    X_test_2 = scaler.transform(X_test)

    mlp = MLPClassifier(hidden_layer_sizes=(10, 10, 10), max_iter=1000)
    mlp.fit(X_train, y_train)
    y_pred = mlp.predict(X_test_2)
    stop = timeit.default_timer()
    time_new = stop - start

    MLP_acc = metrics.accuracy_score(y_test, y_pred)
    MLP_prec = metrics.precision_score(y_test, y_pred, average='weighted')
    MLP_recall = metrics.recall_score(y_test, y_pred, average='weighted')
    MLP_f1_weighted = metrics.f1_score(y_test, y_pred, average='weighted')
    MLP_f1_macro = metrics.f1_score(y_test, y_pred, average='macro')

    y_prob = mlp.predict_proba(X_test_2)[:, 1]
    macro_roc_auc_ovo = roc_auc_score(y_test, y_prob)
    auprc = average_precision_score(y_test, y_prob)

    check = [MLP_acc, MLP_prec, MLP_recall, MLP_f1_weighted, MLP_f1_macro,
             macro_roc_auc_ovo, auprc, time_new]
    return check


##########################  KNN Classifier  ################################
def knn_fun(X_train, y_train, X_test, y_test):

    start = timeit.default_timer()
    classifier = KNeighborsClassifier(n_neighbors=5)
    classifier.fit(X_train, y_train)
    y_pred = classifier.predict(X_test)
    stop = timeit.default_timer()
    time_new = stop - start

    knn_acc = metrics.accuracy_score(y_test, y_pred)
    knn_prec = metrics.precision_score(y_test, y_pred, average='weighted')
    knn_recall = metrics.recall_score(y_test, y_pred, average='weighted')
    knn_f1_weighted = metrics.f1_score(y_test, y_pred, average='weighted')
    knn_f1_macro = metrics.f1_score(y_test, y_pred, average='macro')

    y_prob = classifier.predict_proba(X_test)[:, 1]
    macro_roc_auc_ovo = roc_auc_score(y_test, y_prob)
    auprc = average_precision_score(y_test, y_prob)

    check = [knn_acc, knn_prec, knn_recall, knn_f1_weighted, knn_f1_macro,
             macro_roc_auc_ovo, auprc, time_new]
    return check


##########################  Random Forest Classifier  ################################
def rf_fun(X_train, y_train, X_test, y_test):
    from sklearn.ensemble import RandomForestClassifier
    start = timeit.default_timer()
    rf = RandomForestClassifier(n_estimators=100, class_weight='balanced')
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    stop = timeit.default_timer()
    time_new = stop - start

    fr_acc = metrics.accuracy_score(y_test, y_pred)
    fr_prec = metrics.precision_score(y_test, y_pred, average='weighted')
    fr_recall = metrics.recall_score(y_test, y_pred, average='weighted')
    fr_f1_weighted = metrics.f1_score(y_test, y_pred, average='weighted')
    fr_f1_macro = metrics.f1_score(y_test, y_pred, average='macro')

    y_prob = rf.predict_proba(X_test)[:, 1]
    macro_roc_auc_ovo = roc_auc_score(y_test, y_prob)
    auprc = average_precision_score(y_test, y_prob)

    check = [fr_acc, fr_prec, fr_recall, fr_f1_weighted, fr_f1_macro,
             macro_roc_auc_ovo, auprc, time_new]
    return check


##########################  Logistic Regression Classifier  ################################
def lr_fun(X_train, y_train, X_test, y_test):
    X_train = preprocessing.scale(X_train)
    X_test = preprocessing.scale(X_test)

    start = timeit.default_timer()
    model = LogisticRegression(solver='liblinear', random_state=0,
                               class_weight='balanced')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    stop = timeit.default_timer()
    time_new = stop - start

    LR_acc = metrics.accuracy_score(y_test, y_pred)
    LR_prec = metrics.precision_score(y_test, y_pred, average='weighted')
    LR_recall = metrics.recall_score(y_test, y_pred, average='weighted')
    LR_f1_weighted = metrics.f1_score(y_test, y_pred, average='weighted')
    LR_f1_macro = metrics.f1_score(y_test, y_pred, average='macro')

    y_prob = model.predict_proba(X_test)[:, 1]
    macro_roc_auc_ovo = roc_auc_score(y_test, y_prob)
    auprc = average_precision_score(y_test, y_prob)

    check = [LR_acc, LR_prec, LR_recall, LR_f1_weighted, LR_f1_macro,
             macro_roc_auc_ovo, auprc, time_new]
    return check


##########################  Decision Tree Classifier  ################################
def fun_decision_tree(X_train, y_train, X_test, y_test):
    from sklearn import tree
    start = timeit.default_timer()
    clf = tree.DecisionTreeClassifier(class_weight='balanced')
    clf = clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    stop = timeit.default_timer()
    time_new = stop - start

    dt_acc = metrics.accuracy_score(y_test, y_pred)
    dt_prec = metrics.precision_score(y_test, y_pred, average='weighted')
    dt_recall = metrics.recall_score(y_test, y_pred, average='weighted')
    dt_f1_weighted = metrics.f1_score(y_test, y_pred, average='weighted')
    dt_f1_macro = metrics.f1_score(y_test, y_pred, average='macro')

    y_prob = clf.predict_proba(X_test)[:, 1]
    macro_roc_auc_ovo = roc_auc_score(y_test, y_prob)
    auprc = average_precision_score(y_test, y_prob)

    check = [dt_acc, dt_prec, dt_recall, dt_f1_weighted, dt_f1_macro,
             macro_roc_auc_ovo, auprc, time_new]
    return check


In [53]:
X = final_boruta_data.to_numpy()
y = labels.values

svm_table = []
gauu_nb_table = []
mlp_table = []
knn_table = []
rf_table = []
lr_table = []
dt_table = []

total_splits = 5

sss = StratifiedShuffleSplit(n_splits=total_splits, test_size=0.3, random_state=42)
sss.get_n_splits(X, y)

for splits_ind, (train_index, test_index) in enumerate(sss.split(X, y)):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    start = timeit.default_timer()
    gauu_nb_return = gaus_nb_fun(X_train, y_train, X_test, y_test)
    stop = timeit.default_timer()
    print("NB Time : ", stop - start)

    start = timeit.default_timer()
    mlp_return = mlp_fun(X_train, y_train, X_test, y_test)
    stop = timeit.default_timer()
    print("MLP Time : ", stop - start)

    start = timeit.default_timer()
    knn_return = knn_fun(X_train, y_train, X_test, y_test)
    stop = timeit.default_timer()
    print("KNN Time : ", stop - start)

    start = timeit.default_timer()
    rf_return = rf_fun(X_train, y_train, X_test, y_test)
    stop = timeit.default_timer()
    print("RF Time : ", stop - start)

    start = timeit.default_timer()
    lr_return = lr_fun(X_train, y_train, X_test, y_test)
    stop = timeit.default_timer()
    print("LR Time : ", stop - start)

    start = timeit.default_timer()
    dt_return = fun_decision_tree(X_train, y_train, X_test, y_test)
    stop = timeit.default_timer()
    print("DT Time : ", stop - start)

    start = timeit.default_timer()
    svm_return = svm_fun(X_train, y_train, X_test, y_test)
    stop = timeit.default_timer()
    print("SVM Time : ", stop - start)

    gauu_nb_table.append(gauu_nb_return)
    mlp_table.append(mlp_return)
    knn_table.append(knn_return)
    rf_table.append(rf_return)
    lr_table.append(lr_return)
    dt_table.append(dt_return)
    svm_table.append(svm_return)

col_names = ["Accuracy", "Precision", "Recall", "F1 (weighted)", "F1 (Macro)", "ROC AUC", "AUPRC", "Runtime"]

svm_table_final = DataFrame(svm_table, columns=col_names)
gauu_nb_table_final = DataFrame(gauu_nb_table, columns=col_names)
mlp_table_final = DataFrame(mlp_table, columns=col_names)
knn_table_final = DataFrame(knn_table, columns=col_names)
rf_table_final = DataFrame(rf_table, columns=col_names)
lr_table_final = DataFrame(lr_table, columns=col_names)
dt_table_final = DataFrame(dt_table, columns=col_names)

final_mean_mat = []

final_mean_mat.append(np.transpose((list(svm_table_final.mean()))))
final_mean_mat.append(np.transpose((list(gauu_nb_table_final.mean()))))
final_mean_mat.append(np.transpose((list(mlp_table_final.mean()))))
final_mean_mat.append(np.transpose((list(knn_table_final.mean()))))
final_mean_mat.append(np.transpose((list(rf_table_final.mean()))))
final_mean_mat.append(np.transpose((list(lr_table_final.mean()))))
final_mean_mat.append(np.transpose((list(dt_table_final.mean()))))

final_avg_mat = DataFrame(final_mean_mat, columns=col_names,
                          index=["SVM", "NB", "MLP", "KNN", "RF", "LR", "DT"])

final_avg_mat

NB Time :  0.05041870003333315
MLP Time :  12.521341899991967
KNN Time :  0.6416456999722868
RF Time :  3.1181220000144094
LR Time :  0.08747619995847344
DT Time :  0.22897749999538064
SVM Time :  98.16718210000545
NB Time :  0.054222699953243136
MLP Time :  15.705417899996974
KNN Time :  0.4681513999821618
RF Time :  3.142410699976608
LR Time :  0.09117679996415973
DT Time :  0.23062089999439195
SVM Time :  90.3440244999947
NB Time :  0.0315730000147596
MLP Time :  10.914975900028367
KNN Time :  0.45445069996640086
RF Time :  3.286893600015901
LR Time :  0.10977740003727376
DT Time :  0.2251224999781698
SVM Time :  95.0708393000532
NB Time :  0.039109100005589426
MLP Time :  15.027007099997718
KNN Time :  0.5006590000120923
RF Time :  3.6559996000141837
LR Time :  0.12603440001839772
DT Time :  0.2419508000020869
SVM Time :  106.1130140000023
NB Time :  0.03199940000195056
MLP Time :  10.27718219999224
KNN Time :  0.7180016000056639
RF Time :  3.3679800999816507
LR Time :  0.081975200

,Accuracy,Precision,Recall,F1 (weighted),F1 (Macro),ROC AUC,AUPRC,Runtime
SVM,0.809488,0.945533,0.809488,0.860555,0.584611,0.865245,0.306707,102.182883
NB,0.857353,0.940504,0.857353,0.890401,0.609689,0.844900,0.236233,0.016233
MLP,0.951411,0.938504,0.951411,0.941804,0.634734,0.859888,0.319503,12.870307
KNN,0.949663,0.918171,0.949663,0.928564,0.504745,0.619385,0.083355,0.273707
RF,0.952884,0.938792,0.952884,0.935353,0.559600,0.880707,0.361560,3.183106
LR,0.813758,0.945910,0.813758,0.863383,0.588530,0.866630,0.318276,0.056864
DT,0.927166,0.925708,0.927166,0.926419,0.597870,0.595803,0.092363,0.215056


In [54]:
#taking average of all k-fold performance values
final_mean_mat = []

final_mean_mat.append(np.transpose((list(svm_table_final.std()))))
final_mean_mat.append(np.transpose((list(gauu_nb_table_final.std()))))
final_mean_mat.append(np.transpose((list(mlp_table_final.std()))))
final_mean_mat.append(np.transpose((list(knn_table_final.std()))))
final_mean_mat.append(np.transpose((list(rf_table_final.std()))))
final_mean_mat.append(np.transpose((list(lr_table_final.std()))))
final_mean_mat.append(np.transpose((list(dt_table_final.std()))))

final_avg_mat = DataFrame(final_mean_mat,columns=["Accuracy","Precision","Recall",
                                                    "F1 (weighted)","F1 (Macro)","ROC AUC","AUPRC","Runtime"],
                          index=["SVM","NB","MLP","KNN","RF","LR","DT"])

# final_avg_mat = DataFrame(final_mean_mat,columns=["Accuracy","Precision", "NPV","Sensitivity", "Specificity",
#                                                     "MCC","Recall",
#                                                     "F1 (weighted)","F1 (Macro)","ROC AUC", "ROC-PR","Runtime"],
#                           index=["SVM","NB","MLP","KNN","RF","LR","DT"])

final_avg_mat

,Accuracy,Precision,Recall,F1 (weighted),F1 (Macro),ROC AUC,AUPRC,Runtime
SVM,0.003427,0.000685,0.003427,0.002224,0.002573,0.004984,0.030467,16.186351
NB,0.002454,0.000864,0.002454,0.001618,0.003657,0.003909,0.011690,0.004235
MLP,0.001548,0.002436,0.001548,0.001931,0.015864,0.004590,0.027955,2.416893
KNN,0.000205,0.002146,0.000205,0.000524,0.005052,0.009825,0.002145,0.054714
RF,0.000890,0.004046,0.000890,0.000862,0.005612,0.005954,0.017528,0.207588
LR,0.003439,0.000632,0.003439,0.002199,0.002096,0.004756,0.025802,0.012937
DT,0.002981,0.001154,0.002981,0.002016,0.006049,0.004031,0.004838,0.006677


# Ridge Regression

In [63]:
ridge_df = pd.read_csv("cohort_all_features_capped.csv")
y = ridge_df["delirium_label"]
X_df = ridge_df.drop(columns="delirium_label")
ridge_data = X_df

X_train, X_test, y_train, y_test = train_test_split(
    ridge_data, y,
    test_size=0.3,
    random_state=0)
X_train.shape, X_test.shape

scaler = StandardScaler()
scaler.fit(X_train.fillna(0))

# L1 = Lasso, L2 = Ridge
sel_ = SelectFromModel(LogisticRegression(C=1, penalty='l2', solver='liblinear'))
sel_.fit(scaler.transform(X_train.fillna(0)), y_train)
sel_.get_support()

selected_feat_ridge = X_train.columns[(sel_.get_support())]
print('total features: {}'.format((X_train.shape[1])))
print('selected features: {}'.format(len(selected_feat_ridge)))
print('selection threshold (mean |coef|): {:.4f}'.format(sel_.threshold_))
print('features dropped (below threshold): {}'.format(
      X_train.shape[1] - len(selected_feat_ridge)))

total features: 280
selected features: 93
selection threshold (mean |coef|): 0.1326
features dropped (below threshold): 187


In [64]:
scaler = StandardScaler()
scaler.fit(X_train.fillna(0))

# L1 = Lasso, L2 = Ridge
sel_ = SelectFromModel(LogisticRegression(C=1, penalty='l2', solver='liblinear'))
sel_.fit(scaler.transform(X_train.fillna(0)), y_train)

sel_.get_support()

selected_feat_ridge = X_train.columns[(sel_.get_support())]
print('total features: {}'.format((X_train.shape[1])))
print('selected features: {}'.format(len(selected_feat_ridge)))
print('features with coefficients shrank to zero: {}'.format(
      np.sum(sel_.estimator_.coef_ == 0)))

total features: 280
selected features: 93
features with coefficients shrank to zero: 1


In [65]:
Ridge_Regression_data = sel_.transform(ridge_data.fillna(0))

C:\Users\cabar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


In [ ]:
X = Ridge_Regression_data
y =  labels.values

svm_table = []
gauu_nb_table = []
mlp_table = []
knn_table = []
rf_table = []
lr_table = []
dt_table = []

total_splits = 5

sss = StratifiedShuffleSplit(n_splits=total_splits, test_size=0.3, random_state=42)
sss.get_n_splits(X, y)

for splits_ind, (train_index, test_index) in enumerate(sss.split(X, y)):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    start = timeit.default_timer()
    gauu_nb_return = gaus_nb_fun(X_train, y_train, X_test, y_test)
    stop = timeit.default_timer()
    print("NB Time : ", stop - start)

    start = timeit.default_timer()
    mlp_return = mlp_fun(X_train, y_train, X_test, y_test)
    stop = timeit.default_timer()
    print("MLP Time : ", stop - start)

    start = timeit.default_timer()
    knn_return = knn_fun(X_train, y_train, X_test, y_test)
    stop = timeit.default_timer()
    print("KNN Time : ", stop - start)

    start = timeit.default_timer()
    rf_return = rf_fun(X_train, y_train, X_test, y_test)
    stop = timeit.default_timer()
    print("RF Time : ", stop - start)

    start = timeit.default_timer()
    lr_return = lr_fun(X_train, y_train, X_test, y_test)
    stop = timeit.default_timer()
    print("LR Time : ", stop - start)

    start = timeit.default_timer()
    dt_return = fun_decision_tree(X_train, y_train, X_test, y_test)
    stop = timeit.default_timer()
    print("DT Time : ", stop - start)

    start = timeit.default_timer()
    svm_return = svm_fun(X_train, y_train, X_test, y_test)
    stop = timeit.default_timer()
    print("SVM Time : ", stop - start)

    gauu_nb_table.append(gauu_nb_return)
    mlp_table.append(mlp_return)
    knn_table.append(knn_return)
    rf_table.append(rf_return)
    lr_table.append(lr_return)
    dt_table.append(dt_return)
    svm_table.append(svm_return)

col_names = ["Accuracy", "Precision", "Recall", "F1 (weighted)", "F1 (Macro)", "ROC AUC", "AUPRC", "Runtime"]

svm_table_final = DataFrame(svm_table, columns=col_names)
gauu_nb_table_final = DataFrame(gauu_nb_table, columns=col_names)
mlp_table_final = DataFrame(mlp_table, columns=col_names)
knn_table_final = DataFrame(knn_table, columns=col_names)
rf_table_final = DataFrame(rf_table, columns=col_names)
lr_table_final = DataFrame(lr_table, columns=col_names)
dt_table_final = DataFrame(dt_table, columns=col_names)

final_mean_mat = []

final_mean_mat.append(np.transpose((list(svm_table_final.mean()))))
final_mean_mat.append(np.transpose((list(gauu_nb_table_final.mean()))))
final_mean_mat.append(np.transpose((list(mlp_table_final.mean()))))
final_mean_mat.append(np.transpose((list(knn_table_final.mean()))))
final_mean_mat.append(np.transpose((list(rf_table_final.mean()))))
final_mean_mat.append(np.transpose((list(lr_table_final.mean()))))
final_mean_mat.append(np.transpose((list(dt_table_final.mean()))))

final_avg_mat = DataFrame(final_mean_mat, columns=col_names,
                          index=["SVM", "NB", "MLP", "KNN", "RF", "LR", "DT"])

final_avg_mat

NB Time :  0.08579810004448518


C:\Users\cabar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


MLP Time :  9.508439800003543
KNN Time :  1.0812648000428453
RF Time :  10.759792999946512
LR Time :  1.907804100017529
DT Time :  1.7010316000087187


In [68]:
#taking average of all k-fold performance values
final_mean_mat = []

final_mean_mat.append(np.transpose((list(svm_table_final.std()))))
final_mean_mat.append(np.transpose((list(gauu_nb_table_final.std()))))
final_mean_mat.append(np.transpose((list(mlp_table_final.std()))))
final_mean_mat.append(np.transpose((list(knn_table_final.std()))))
final_mean_mat.append(np.transpose((list(rf_table_final.std()))))
final_mean_mat.append(np.transpose((list(lr_table_final.std()))))
final_mean_mat.append(np.transpose((list(dt_table_final.std()))))

final_avg_mat = DataFrame(final_mean_mat,columns=["Accuracy","Precision","Recall",
                                                    "F1 (weighted)","F1 (Macro)","ROC AUC","AUPRC","Runtime"],
                          index=["SVM","NB","MLP","KNN","RF","LR","DT"])

# final_avg_mat = DataFrame(final_mean_mat,columns=["Accuracy","Precision", "NPV","Sensitivity", "Specificity",
#                                                     "MCC","Recall",
#                                                     "F1 (weighted)","F1 (Macro)","ROC AUC", "ROC-PR","Runtime"],
#                           index=["SVM","NB","MLP","KNN","RF","LR","DT"])

final_avg_mat

,Accuracy,Precision,Recall,F1 (weighted),F1 (Macro),ROC AUC,AUPRC,Runtime
SVM,0.003242,0.001532,0.003242,0.002239,0.005322,0.004828,0.025039,129.445947
NB,0.012560,0.001416,0.012560,0.009062,0.005862,0.009208,0.008326,0.016585
MLP,0.002919,0.000778,0.002919,0.001186,0.006053,0.016596,0.009938,4.251777
KNN,0.000652,0.006930,0.000652,0.000765,0.005255,0.012381,0.004136,0.064974
RF,0.000227,0.005047,0.000227,0.000376,0.003551,0.008845,0.024163,0.909796
LR,0.001742,0.001292,0.001742,0.001228,0.003720,0.004649,0.021427,0.422968
DT,0.001754,0.001261,0.001754,0.000461,0.006678,0.010160,0.005216,0.232648


# Kernel Trick

In [7]:
# Build RFF features
from sklearn.kernel_approximation import RBFSampler

df = pd.read_csv("cohort_all_features_capped.csv")
labels = df["delirium_label"]
X_df = df.drop(columns="delirium_label")

X_full = X_df.fillna(0).to_numpy()

rbf_feature = RBFSampler(gamma=1, n_components=256, random_state=42)
RFF_data = rbf_feature.fit_transform(X_full)

In [8]:
X = RFF_data
y = labels.values

svm_table = []
gauu_nb_table = []
mlp_table = []
knn_table = []
rf_table = []
lr_table = []
dt_table = []

total_splits = 5

sss = StratifiedShuffleSplit(n_splits=total_splits, test_size=0.3, random_state=42)
sss.get_n_splits(X, y)

for splits_ind, (train_index, test_index) in enumerate(sss.split(X, y)):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    start = timeit.default_timer()
    gauu_nb_return = gaus_nb_fun(X_train, y_train, X_test, y_test)
    stop = timeit.default_timer()
    print("NB Time : ", stop - start)

    start = timeit.default_timer()
    mlp_return = mlp_fun(X_train, y_train, X_test, y_test)
    stop = timeit.default_timer()
    print("MLP Time : ", stop - start)

    start = timeit.default_timer()
    knn_return = knn_fun(X_train, y_train, X_test, y_test)
    stop = timeit.default_timer()
    print("KNN Time : ", stop - start)

    start = timeit.default_timer()
    rf_return = rf_fun(X_train, y_train, X_test, y_test)
    stop = timeit.default_timer()
    print("RF Time : ", stop - start)

    start = timeit.default_timer()
    lr_return = lr_fun(X_train, y_train, X_test, y_test)
    stop = timeit.default_timer()
    print("LR Time : ", stop - start)

    start = timeit.default_timer()
    dt_return = fun_decision_tree(X_train, y_train, X_test, y_test)
    stop = timeit.default_timer()
    print("DT Time : ", stop - start)

    start = timeit.default_timer()
    svm_return = svm_fun(X_train, y_train, X_test, y_test)
    stop = timeit.default_timer()
    print("SVM Time : ", stop - start)

    gauu_nb_table.append(gauu_nb_return)
    mlp_table.append(mlp_return)
    knn_table.append(knn_return)
    rf_table.append(rf_return)
    lr_table.append(lr_return)
    dt_table.append(dt_return)
    svm_table.append(svm_return)

col_names = ["Accuracy", "Precision", "Recall", "F1 (weighted)", "F1 (Macro)", "ROC AUC", "AUPRC", "Runtime"]

svm_table_final = DataFrame(svm_table, columns=col_names)
gauu_nb_table_final = DataFrame(gauu_nb_table, columns=col_names)
mlp_table_final = DataFrame(mlp_table, columns=col_names)
knn_table_final = DataFrame(knn_table, columns=col_names)
rf_table_final = DataFrame(rf_table, columns=col_names)
lr_table_final = DataFrame(lr_table, columns=col_names)
dt_table_final = DataFrame(dt_table, columns=col_names)

final_mean_mat = []

final_mean_mat.append(np.transpose((list(svm_table_final.mean()))))
final_mean_mat.append(np.transpose((list(gauu_nb_table_final.mean()))))
final_mean_mat.append(np.transpose((list(mlp_table_final.mean()))))
final_mean_mat.append(np.transpose((list(knn_table_final.mean()))))
final_mean_mat.append(np.transpose((list(rf_table_final.mean()))))
final_mean_mat.append(np.transpose((list(lr_table_final.mean()))))
final_mean_mat.append(np.transpose((list(dt_table_final.mean()))))

final_avg_mat = DataFrame(final_mean_mat, columns=col_names,
                          index=["SVM", "NB", "MLP", "KNN", "RF", "LR", "DT"])

final_avg_mat

C:\Users\cabar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


NB Time :  0.26088870000057796
MLP Time :  6.146394200000032
KNN Time :  2.183854699999756


C:\Users\cabar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


RF Time :  50.100586999999905
LR Time :  0.44667930000014167
DT Time :  16.836792799999785
SVM Time :  0.5993897000007564


C:\Users\cabar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...

C:\Users\cabar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


NB Time :  0.20195409999996627
MLP Time :  6.363914900000054
KNN Time :  2.240248199999769


C:\Users\cabar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


RF Time :  53.05434849999983
LR Time :  0.4369094000003315
DT Time :  26.961921999999504
SVM Time :  0.5891674999993484


,Accuracy,Precision,Recall,F1 (weighted),F1 (Macro),ROC AUC,AUPRC,Runtime
SVM,0.586567,0.907475,0.586567,0.701436,0.409610,0.496063,0.050603,0.340501
NB,0.951436,0.905230,0.951436,0.927758,0.487557,0.498965,0.050893,0.139268
MLP,0.897628,0.907410,0.897628,0.902454,0.498731,0.502302,0.049259,7.252774
KNN,0.950637,0.906602,0.950637,0.927407,0.487852,0.513605,0.050076,1.077545
RF,0.951436,0.905230,0.951436,0.927758,0.487557,0.504175,0.051688,51.395682
LR,0.587166,0.907528,0.587166,0.701907,0.409905,0.495990,0.050611,0.252579
DT,0.899051,0.907374,0.899051,0.903168,0.498606,0.498817,0.048607,20.881267


In [48]:
#taking average of all k-fold performance values
final_mean_mat = []

final_mean_mat.append(np.transpose((list(svm_table_final.std()))))
final_mean_mat.append(np.transpose((list(gauu_nb_table_final.std()))))
final_mean_mat.append(np.transpose((list(mlp_table_final.std()))))
final_mean_mat.append(np.transpose((list(knn_table_final.std()))))
final_mean_mat.append(np.transpose((list(rf_table_final.std()))))
final_mean_mat.append(np.transpose((list(lr_table_final.std()))))
final_mean_mat.append(np.transpose((list(dt_table_final.std()))))

final_avg_mat = DataFrame(final_mean_mat,columns=["Accuracy","Precision","Recall",
                                                    "F1 (weighted)","F1 (Macro)","ROC AUC","AUPRC","Runtime"],
                          index=["SVM","NB","MLP","KNN","RF","LR","DT"])

# final_avg_mat = DataFrame(final_mean_mat,columns=["Accuracy","Precision", "NPV","Sensitivity", "Specificity",
#                                                     "MCC","Recall",
#                                                     "F1 (weighted)","F1 (Macro)","ROC AUC", "ROC-PR","Runtime"],
#                           index=["SVM","NB","MLP","KNN","RF","LR","DT"])

final_avg_mat

,Accuracy,Precision,Recall,F1 (weighted),F1 (Macro),F1 (Micro),ROC AUC
NB,0.932977,0.870446,0.932977,0.900627,0.321775,0.932977,0.500000
MLP,0.932873,0.886083,0.932873,0.900711,0.322601,0.932873,0.500323
KNN,0.926587,0.908187,0.926587,0.915659,0.448742,0.926587,0.580523
RF,0.941485,0.930270,0.941485,0.933009,0.557534,0.941485,0.640535
LR,0.932977,0.870446,0.932977,0.900627,0.321775,0.932977,0.500000
DT,0.910821,0.916018,0.910821,0.913331,0.513490,0.910821,0.654679


# Keras Classifier